# ARC-Challenge — CoT Baseline Evaluation
## Chain-of-Thought vs No-Guidance Baseline

**Purpose:** Runs TWO conditions on the same 300 ARC-Challenge questions (seed=42):
- **CoT:** Qwen 1.5B × 5 votes with `Let's think step by step`
- **Baseline:** Qwen 1.5B × 5 votes, no CoT, no guide

Compare against **Guided pipeline** results from the original notebook.

| Condition | Compute | Description |
|---|---|---|
| Baseline | 7.5B | No guide, no CoT |
| **CoT (this notebook)** | **7.5B** | No guide, 'think step by step' |
| Guided (original notebook) | 10.5B | Fine-tuned 3B guide + 1.5B solver |

**Why ARC-Challenge?** ARC-Challenge contains science exam questions specifically selected to resist simple retrieval and co-occurrence heuristics. Both correct and incorrect answers are plausible, making genuine reasoning essential — an ideal setting to measure CoT benefit.

**Task format:** Given a science question and 4 labelled answer choices (A/B/C/D), the model must output the letter of the correct answer.

**Same seed=42, same 300 questions — direct 3-way comparison is valid.**

In [1]:
# CELL 1 -- Install
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [2]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login

login("")  # paste your token here
print("HuggingFace login done")

HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU check
import os, json, re, random, time
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/arc_challenge_cot_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")

PyTorch : 2.9.0+cu126
GPU     : Tesla P100-PCIE-16GB
VRAM    : 17.1 GB
Output  : /kaggle/working/arc_challenge_cot_eval


In [4]:
# CELL 4 -- Configuration
# NOTE: No guide model. Only the 1.5B solver.
# CRITICAL: max_eval_samples=300 and random_seed=42 MUST match original notebook.
CONFIG = {
    "solver_model"     : "Qwen/Qwen2.5-1.5B-Instruct",
    "dataset_name"     : "allenai/ai2_arc",
    "dataset_config"   : "ARC-Challenge",
    "dataset_split"    : "test",
    "max_eval_samples" : 900,   # MUST match original notebook
    "random_seed"      : 42,    # MUST match original notebook
    "n_votes"          : 5,
    "vote_temperature" : 0.4,   # MUST match original notebook
    "max_new_tokens"   : 250,   # ARC needs some reasoning but answers are letters
    "solver_params_B"  : 1.5,
    "results_file"     : f"{OUTPUT_DIR}/results.jsonl",
    "angle1_file"      : f"{OUTPUT_DIR}/angle1_compute.json",
    "angle2_file"      : f"{OUTPUT_DIR}/angle2_consistency.json",
    "angle3_file"      : f"{OUTPUT_DIR}/angle3_calibration.json",
    "angle4_file"      : f"{OUTPUT_DIR}/angle4_answer_position_bias.json",
    "checkpoint_file"  : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"       : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")

Config ready:
  solver_model            : Qwen/Qwen2.5-1.5B-Instruct
  dataset_name            : allenai/ai2_arc
  dataset_config          : ARC-Challenge
  dataset_split           : test
  max_eval_samples        : 900
  random_seed             : 42
  n_votes                 : 5
  vote_temperature        : 0.4
  max_new_tokens          : 250
  solver_params_B         : 1.5
  results_file            : /kaggle/working/arc_challenge_cot_eval/results.jsonl
  angle1_file             : /kaggle/working/arc_challenge_cot_eval/angle1_compute.json
  angle2_file             : /kaggle/working/arc_challenge_cot_eval/angle2_consistency.json
  angle3_file             : /kaggle/working/arc_challenge_cot_eval/angle3_calibration.json
  angle4_file             : /kaggle/working/arc_challenge_cot_eval/angle4_answer_position_bias.json
  checkpoint_file         : /kaggle/working/arc_challenge_cot_eval/checkpoint.json
  save_every              : 25


In [5]:
# CELL 5 -- Load ARC-Challenge dataset
# ARC-Challenge fields:
#   question  : str
#   choices   : dict with 'text' (list of str) and 'label' (list of str, e.g. ['A','B','C','D'])
#   answerKey : str  (e.g. 'A', 'B', 'C', 'D' -- occasionally '1','2','3','4' in older items)
# We normalise answerKey to always be A/B/C/D.
# Same seed=42 and max_eval_samples=300 guarantees the same 300 questions as original.

# Numeric label fallback (some ARC items use '1'->'A' etc.)
NUM_TO_LETTER = {"1": "A", "2": "B", "3": "C", "4": "D", "5": "E"}

print("Loading ARC-Challenge...")
raw_ds = load_dataset(CONFIG["dataset_name"], CONFIG["dataset_config"])
print(f"Splits     : {list(raw_ds.keys())}")
print(f"Test size  : {len(raw_ds[CONFIG['dataset_split']])}")


def normalise_arc(item):
    """Normalise an ARC item into the standard evaluation format.

    choices_map: dict mapping label -> choice text (e.g. {'A': 'Photosynthesis', ...})
    answer     : normalised letter string, always in {A, B, C, D, E}
    answer_pos : 0-based index of the correct answer in the choices list (for Angle 4)
    """
    labels = item["choices"]["label"]
    texts  = item["choices"]["text"]
    choices_map = {lbl: txt for lbl, txt in zip(labels, texts)}

    raw_key = str(item["answerKey"]).strip()
    answer  = NUM_TO_LETTER.get(raw_key, raw_key).upper()

    # answer_pos: position index (0=A, 1=B, 2=C, 3=D) for position-bias analysis
    norm_labels = [NUM_TO_LETTER.get(l, l).upper() for l in labels]
    answer_pos  = norm_labels.index(answer) if answer in norm_labels else -1

    return {
        "question"   : item["question"].strip(),
        "choices_map": choices_map,
        "labels"     : labels,
        "texts"      : texts,
        "answer"     : answer,
        "answer_pos" : answer_pos,   # 0=A, 1=B, 2=C, 3=D
    }


all_data = [normalise_arc(x) for x in raw_ds[CONFIG["dataset_split"]]]

# Fix seed ONCE -- must match original notebook
random.seed(CONFIG["random_seed"])
test_data = random.sample(all_data, CONFIG["max_eval_samples"])

ans_counts = Counter(d["answer"] for d in test_data)
print(f"Sampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
print("  Answer distribution:")
for lbl in sorted(ans_counts):
    print(f"    {lbl}: {ans_counts[lbl]}")
print(f"First Q : {test_data[0]['question'][:80]}")
for lbl, txt in test_data[0]['choices_map'].items():
    print(f"  ({lbl}) {txt}")
print(f"  Answer: {test_data[0]['answer']}")
print("ARC-Challenge loaded")

Loading ARC-Challenge...


README.md: 0.00B [00:00, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…):   0%|          | 0.00/190k [00:00<?, ?B/s]

ARC-Challenge/test-00000-of-00001.parque(…):   0%|          | 0.00/204k [00:00<?, ?B/s]

ARC-Challenge/validation-00000-of-00001.(…):   0%|          | 0.00/55.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

Splits     : ['train', 'test', 'validation']
Test size  : 1172
Sampled 900 questions (seed=42)
  Answer distribution:
    A: 195
    B: 255
    C: 226
    D: 224
First Q : When cold temperatures are produced in a chemical reaction, the reaction is know
  (A) exothermic.
  (B) endothermic.
  (C) suspension.
  (D) vaporization.
  Answer: B
ARC-Challenge loaded


In [6]:
# CELL 6 -- Answer extraction
# ARC-Challenge is a 4-way multiple-choice task: model must output A, B, C, or D.
# GT answer is always a single uppercase letter.

VALID_LABELS = {"A", "B", "C", "D", "E"}  # E exists in a small number of ARC items


def extract_gt_answer(answer_str):
    """GT answer is already a normalised letter from normalise_arc."""
    return str(answer_str).strip().upper()


def extract_pred_answer(text):
    """Extract a letter answer (A/B/C/D/E) from model output.

    Tries patterns in order of precision:
      1. #### A  (explicit marker, matches system prompt convention)
      2. 'the answer is A' / 'answer: A'
      3. '(A)' or '**A**' or 'Answer: A'
      4. 'option A' / 'choice A'
      5. Standalone capital letter at end of text
    Returns '' if none found.
    """
    t = text.strip()

    # #### marker
    m = re.search(r"####\s*([A-E])\b", t, re.IGNORECASE)
    if m: return m.group(1).upper()

    # 'the answer is A' / 'answer: A' / 'answer is (A)'
    m = re.search(r"(?:the\s+)?answer\s*(?:is)?\s*:?\s*\(?([A-E])\)?", t, re.IGNORECASE)
    if m: return m.group(1).upper()

    # **(A)** or **A** (bold markdown)
    m = re.search(r"\*\*\(?([A-E])\)?\*\*", t, re.IGNORECASE)
    if m: return m.group(1).upper()

    # 'option A' / 'choice A' / 'select A'
    m = re.search(r"(?:option|choice|select|choose)\s+\(?([A-E])\)?", t, re.IGNORECASE)
    if m: return m.group(1).upper()

    # '(A)' anywhere -- parenthesised letter
    m = re.search(r"\(([A-E])\)", t, re.IGNORECASE)
    if m: return m.group(1).upper()

    # 'therefore/thus/so, A' at end of reasoning
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+\(?([A-E])\)?", t, re.IGNORECASE)
    if m: return m.group(1).upper()

    # Standalone capital letter at end of text
    m = re.search(r"\b([A-E])\s*$", t)
    if m: return m.group(1).upper()

    return ""


# Self-test
_tests = [
    ("#### A", "A"),
    ("#### C", "C"),
    ("The answer is B", "B"),
    ("answer: D", "D"),
    ("The answer is (C)", "C"),
    ("**B**", "B"),
    ("**(A)**", "A"),
    ("Option D is correct.", "D"),
    ("Choose (C) because...", "C"),
    ("(B)", "B"),
    ("Therefore, A", "A"),
    ("Some irrelevant text.", ""),
]
all_ok = all(extract_pred_answer(t) == e for t, e in _tests)
print("Extractor:", "ALL PASSED" if all_ok else "FAILURES -- fix before running eval")
if not all_ok:
    for t, e in _tests:
        got = extract_pred_answer(t)
        if got != e:
            print(f"  FAIL: input={repr(t)}  expected={repr(e)}  got={repr(got)}")

Extractor: ALL PASSED


In [7]:
# CELL 7 -- Load solver model (Qwen 1.5B only -- no guide model needed)
# T4 has 15GB. 1.5B in float16 ~ 3GB. Plenty of headroom.

print(f"Loading: {CONFIG['solver_model']}")
solver_tok = AutoTokenizer.from_pretrained(CONFIG["solver_model"])
if solver_tok.pad_token is None:
    solver_tok.pad_token = solver_tok.eos_token

solver_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["solver_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

vram = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used : {vram:.2f} GB")
print(f"Headroom  : {15.0 - vram:.1f} GB")
print("Solver ready")

Loading: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

VRAM used : 3.09 GB
Headroom  : 11.9 GB
Solver ready


In [8]:
# CELL 8 -- Prompts and generation functions
#
# COT_SYSTEM      -- adds 'Let's think step by step' (the new CoT condition)
# BASELINE_SYSTEM -- no CoT, no guide  (matches original notebook baseline)
# REFINER_SYSTEM  -- tie-breaker (same as original)
#
# ARC task: given a science question + 4 labelled choices, output the correct letter.

COT_SYSTEM = (
    "You are an expert science exam solver.\n"
    "Let's think step by step.\n"
    "Read the question carefully, then reason through each answer choice.\n"
    "Eliminate implausible options before committing to the best answer.\n"
    "Keep your reasoning concise and grounded in science.\n"
    "Your FINAL line must be exactly: #### A  (or B, C, D)\n"
    "Do not write anything after the final line.\n\n"
    "Example:\n"
    "Question: What gas do plants absorb from the air during photosynthesis?\n"
    "(A) Oxygen  (B) Nitrogen  (C) Carbon dioxide  (D) Hydrogen\n"
    "Step 1: Photosynthesis converts light energy into glucose.\n"
    "Step 2: The inputs are water, light, and CO2 -- not O2 or N2.\n"
    "Step 3: Plants absorb carbon dioxide (C) and release oxygen.\n"
    "#### C"
)

BASELINE_SYSTEM = (
    "You are an expert science exam solver.\n"
    "Read the question carefully and choose the best answer.\n"
    "Your FINAL line must be exactly: #### A  (or B, C, D)"
)

REFINER_SYSTEM = (
    "You are a careful science exam solver.\n"
    "Previous attempts gave different answers. Ignore all of them.\n"
    "Re-read the question and choices from scratch, then decide.\n"
    "Your FINAL line must be exactly: #### A  (or B, C, D)"
)


def format_arc_question(item):
    """Format an ARC item as a multiple-choice question string."""
    lines = [f"Question: {item['question']}"]
    for lbl, txt in zip(item["labels"], item["texts"]):
        norm_lbl = NUM_TO_LETTER.get(lbl, lbl).upper()
        lines.append(f"({norm_lbl}) {txt}")
    lines.append("Which answer choice is correct? Reply with the letter only.")
    return "\n".join(lines)


def run_solver(messages, temperature):
    prompt = solver_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = solver_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(solver_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = solver_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=max(temperature, 0.05),
            do_sample=True,
            top_p=0.92,
            top_k=40,
            pad_token_id=solver_tok.eos_token_id,
            repetition_penalty=1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return solver_tok.decode(new_toks, skip_special_tokens=True).strip()


def generate_cot(item):
    """CoT: Let's think step by step -- no guide."""
    return run_solver(
        [{"role": "system", "content": COT_SYSTEM},
         {"role": "user",   "content": format_arc_question(item)}],
        temperature=CONFIG["vote_temperature"],
    )


def generate_baseline(item):
    """Baseline: no CoT, no guide."""
    return run_solver(
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": format_arc_question(item)}],
        temperature=CONFIG["vote_temperature"],
    )


def generate_refiner(item, candidates):
    """Tie-breaker: re-evaluate from scratch ignoring previous votes."""
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"{format_arc_question(item)}\n\n"
        f"Previous attempts gave: {cands}\n"
        "Ignore all previous attempts. Re-evaluate from scratch:"
    )
    return run_solver(
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        temperature=0.3,
    )


print("Generation functions ready")
print("  CoT prompt   : 'Let's think step by step'")
print("  Temperature  :", CONFIG["vote_temperature"])

Generation functions ready
  CoT prompt   : 'Let's think step by step'
  Temperature  : 0.4


In [9]:
# CELL 9 -- Voting logic (identical to original notebook, adapted for ARC item dict)

def vote_and_decide(answers, item, gt_answer=None):
    valid = [a for a in answers if a and a.strip()]
    if not valid: valid = answers

    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        ref_raw  = generate_refiner(item, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None
        all_v      = valid + ([ref_ans] if ref_ans else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]
        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted           = total - new_top_c
        vote_counts      = new_counts

    return {
        "final_answer": final, "strategy": strategy,
        "confidence": conf, "vote_counts": dict(vote_counts),
        "correct_votes": correct_votes, "total_votes": total,
        "vote_consistency": round(vote_consistency, 4),
        "wasted_votes": wasted,
        "refiner_used": refiner_used, "refiner_correct": refiner_correct,
    }

print("Voting logic ready")

Voting logic ready


In [10]:
# CELL 10 -- Single question test (verify both conditions work before full run)

item = test_data[0]
gt   = extract_gt_answer(item["answer"])

print("=" * 65)
print(f"Question : {item['question']}")
for lbl, txt in zip(item["labels"], item["texts"]):
    print(f"  ({lbl}) {txt}")
print(f"GT Answer: {gt}")

# CoT
print("\n[CoT] 3 sample votes:")
for i in range(3):
    raw  = generate_cot(item)
    pred = extract_pred_answer(raw)
    print(f"  Vote {i+1}: '{pred}'  | raw[:80]: {raw[:80]}")

# Baseline
print("\n[Baseline] 3 sample votes:")
for i in range(3):
    raw  = generate_baseline(item)
    pred = extract_pred_answer(raw)
    print(f"  Vote {i+1}: '{pred}'")

print("\nSingle test done. Run Cell 11 for full evaluation.")

Question : When cold temperatures are produced in a chemical reaction, the reaction is known as
  (A) exothermic.
  (B) endothermic.
  (C) suspension.
  (D) vaporization.
GT Answer: B

[CoT] 3 sample votes:
  Vote 1: 'B'  | raw[:80]: (B)
  Vote 2: 'B'  | raw[:80]: (B)
  Vote 3: 'B'  | raw[:80]: (B)

[Baseline] 3 sample votes:
  Vote 1: 'B'
  Vote 2: 'B'
  Vote 3: 'B'

Single test done. Run Cell 11 for full evaluation.


In [11]:
# CELL 11 -- Full Dual Evaluation Loop
# Runs all 300 questions under TWO conditions:
#   Mode A: cot      (1.5B x5, 'Let's think step by step')
#   Mode B: baseline (1.5B x5, no CoT)
# Checkpoints every 25 questions.

print(f"Dual evaluation: {len(test_data)} ARC-Challenge questions")
print(f"Each question  : {CONFIG['n_votes']} CoT votes + {CONFIG['n_votes']} baseline votes")
print("-" * 65)

cot_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        cot_results  = [r for r in lines if r.get("mode") == "cot"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from {start_idx} (CoT: {len(cot_results)}, Base: {len(base_results)})")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="ARC-Challenge CoT Eval"):
    item      = test_data[idx]
    gt_answer = extract_gt_answer(item["answer"])
    ans_pos   = item["answer_pos"]  # 0=A, 1=B, 2=C, 3=D (for position-bias analysis)

    # ---- CoT condition ----------------------------------------
    try:
        cot_votes_raw = [extract_pred_answer(generate_cot(item))
                         for _ in range(CONFIG["n_votes"])]
        c_dec = vote_and_decide(cot_votes_raw, item, gt_answer)
        cot_results.append({
            "mode": "cot", "idx": idx,
            "question": item["question"],
            "gt_answer": gt_answer, "answer_pos": ans_pos,
            "final_answer": c_dec["final_answer"],
            "correct": c_dec["final_answer"] == gt_answer,
            "strategy": c_dec["strategy"], "confidence": c_dec["confidence"],
            "correct_votes": c_dec["correct_votes"], "total_votes": c_dec["total_votes"],
            "vote_consistency": c_dec["vote_consistency"],
            "wasted_votes": c_dec["wasted_votes"],
            "refiner_used": c_dec["refiner_used"],
            "refiner_correct": c_dec["refiner_correct"],
            "vote_counts": c_dec["vote_counts"],
        })
    except RuntimeError as e:
        cot_results.append({
            "mode": "cot", "idx": idx,
            "question": item["question"],
            "gt_answer": gt_answer, "answer_pos": ans_pos,
            "final_answer": "", "correct": False, "strategy": "error",
            "confidence": 0.0, "correct_votes": 0,
            "total_votes": CONFIG["n_votes"], "vote_consistency": 0.0,
            "wasted_votes": CONFIG["n_votes"], "refiner_used": False,
            "refiner_correct": None, "vote_counts": {}, "error": str(e),
        })

    # ---- Baseline condition -----------------------------------
    try:
        base_votes_raw = [extract_pred_answer(generate_baseline(item))
                          for _ in range(CONFIG["n_votes"])]
        b_dec = vote_and_decide(base_votes_raw, item, gt_answer)
        base_results.append({
            "mode": "baseline", "idx": idx,
            "question": item["question"],
            "gt_answer": gt_answer, "answer_pos": ans_pos,
            "final_answer": b_dec["final_answer"],
            "correct": b_dec["final_answer"] == gt_answer,
            "strategy": b_dec["strategy"], "confidence": b_dec["confidence"],
            "correct_votes": b_dec["correct_votes"], "total_votes": b_dec["total_votes"],
            "vote_consistency": b_dec["vote_consistency"],
            "wasted_votes": b_dec["wasted_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx,
            "question": item["question"],
            "gt_answer": gt_answer, "answer_pos": ans_pos,
            "final_answer": "", "correct": False, "strategy": "error",
            "confidence": 0.0, "correct_votes": 0,
            "total_votes": CONFIG["n_votes"], "vote_consistency": 0.0,
            "wasted_votes": CONFIG["n_votes"], "refiner_used": False,
            "refiner_correct": None, "vote_counts": {}, "error": str(e),
        })

    # Checkpoint
    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in cot_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] CoT: {c_acc:.1f}%  Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

# Final save
with open(CONFIG["results_file"], "w") as f:
    for r in cot_results + base_results:
        f.write(json.dumps(r) + "\n")

c_c = sum(r["correct"] for r in cot_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nDone. CoT: {c_c}/{len(cot_results)} = {c_c/len(cot_results)*100:.1f}%")
print(f"Baseline : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"CoT vs Baseline: {(c_c/len(cot_results) - b_c/len(base_results))*100:+.1f} pts")

Dual evaluation: 900 ARC-Challenge questions
Each question  : 5 CoT votes + 5 baseline votes
-----------------------------------------------------------------
Starting fresh


ARC-Challenge CoT Eval:   0%|          | 0/900 [00:00<?, ?it/s]

  [ 25] CoT: 68.0%  Baseline: 72.0%  (0.6 min)
  [ 50] CoT: 70.0%  Baseline: 72.0%  (1.3 min)
  [ 75] CoT: 70.7%  Baseline: 70.7%  (1.9 min)
  [100] CoT: 72.0%  Baseline: 72.0%  (2.5 min)
  [125] CoT: 70.4%  Baseline: 71.2%  (3.2 min)
  [150] CoT: 72.0%  Baseline: 72.0%  (3.8 min)
  [175] CoT: 72.6%  Baseline: 71.4%  (4.5 min)
  [200] CoT: 73.5%  Baseline: 71.5%  (5.1 min)
  [225] CoT: 74.2%  Baseline: 72.0%  (5.7 min)
  [250] CoT: 73.6%  Baseline: 72.4%  (6.4 min)
  [275] CoT: 74.2%  Baseline: 73.5%  (7.0 min)
  [300] CoT: 74.3%  Baseline: 73.7%  (7.6 min)
  [325] CoT: 74.5%  Baseline: 74.2%  (8.3 min)
  [350] CoT: 73.7%  Baseline: 73.4%  (8.9 min)
  [375] CoT: 73.1%  Baseline: 72.5%  (9.5 min)
  [400] CoT: 73.5%  Baseline: 73.2%  (10.2 min)
  [425] CoT: 73.2%  Baseline: 72.9%  (10.8 min)
  [450] CoT: 73.6%  Baseline: 73.1%  (11.5 min)
  [475] CoT: 73.7%  Baseline: 73.5%  (12.1 min)
  [500] CoT: 73.6%  Baseline: 73.2%  (12.7 min)
  [525] CoT: 73.5%  Baseline: 73.0%  (13.4 min)
  [550]

In [12]:
# CELL 12 -- ANGLE 1: COMPUTE EFFICIENCY
# Both CoT and Baseline cost 7.5B (1.5B x5). Zero guide overhead.

S, N = CONFIG["solver_params_B"], CONFIG["n_votes"]
compute = S * N  # 7.5B for both

c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
c_wasted = sum(r["wasted_votes"] for r in cot_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)
c_ref = sum(r["refiner_used"] for r in cot_results)
b_ref = sum(r["refiner_used"] for r in base_results)

# Extraction failure rate (model outputs a letter outside A-D or nothing)
c_fail = sum(1 for r in cot_results  if r["final_answer"] not in VALID_LABELS)
b_fail = sum(1 for r in base_results if r["final_answer"] not in VALID_LABELS)

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (ARC-Challenge)")
print("=" * 65)
print(f"\n  {'Setup':<28} | {'Compute':>8} | {'Accuracy':>9}")
print(f"  {'-'*28}-+-{'-'*8}-+-{'-'*9}")
print(f"  {'Baseline (no CoT)':<28} | {compute:>6.1f}B  | {b_acc:>8.1f}%")
print(f"  {'CoT (think step by step)':<28} | {compute:>6.1f}B  | {c_acc:>8.1f}%")
print(f"\n  Accuracy gain (CoT vs Baseline): {c_acc - b_acc:+.1f} pts")
print(f"  Compute difference              : 0 (same 7.5B)")
print(f"  Wasted votes saved              : {b_wasted - c_wasted} ({c_wasted} CoT vs {b_wasted} Baseline)")
print(f"  Refiner: CoT={c_ref}  Baseline={b_ref}")
print(f"  Extraction failures: CoT={c_fail}  Baseline={b_fail}")

angle1 = {
    "dataset": "ARC-Challenge", "n_questions": len(cot_results), "compute_B": compute,
    "cot_accuracy": round(c_acc, 2), "baseline_accuracy": round(b_acc, 2),
    "accuracy_gain": round(c_acc - b_acc, 2),
    "cot_wasted": c_wasted, "baseline_wasted": b_wasted,
    "cot_refiner": c_ref, "baseline_refiner": b_ref,
    "cot_extraction_failures": c_fail, "baseline_extraction_failures": b_fail,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")

ANGLE 1 -- COMPUTE EFFICIENCY  (ARC-Challenge)

  Setup                        |  Compute |  Accuracy
  -----------------------------+----------+----------
  Baseline (no CoT)            |    7.5B  |     71.8%
  CoT (think step by step)     |    7.5B  |     72.1%

  Accuracy gain (CoT vs Baseline): +0.3 pts
  Compute difference              : 0 (same 7.5B)
  Wasted votes saved              : 41 (53 CoT vs 94 Baseline)
  Refiner: CoT=0  Baseline=0
  Extraction failures: CoT=0  Baseline=0

Saved -> /kaggle/working/arc_challenge_cot_eval/angle1_compute.json


In [13]:
# CELL 13 -- ANGLE 2: VOTE CONSISTENCY

c_cons = [r["vote_consistency"] for r in cot_results]
b_cons = [r["vote_consistency"] for r in base_results]
c_mean = np.mean(c_cons)
b_mean = np.mean(b_cons)
lift   = c_mean / max(b_mean, 1e-6)

cot_wins      = sum(1 for c, b in zip(c_cons, b_cons) if c > b)
baseline_wins = sum(1 for c, b in zip(c_cons, b_cons) if b > c)
tied          = sum(1 for c, b in zip(c_cons, b_cons) if c == b)

def bucket(scores):
    return {
        "all_wrong (0%)":    sum(1 for s in scores if s == 0.0),
        "low (1-39%)":       sum(1 for s in scores if 0.0 < s < 0.4),
        "medium (40-79%)":   sum(1 for s in scores if 0.4 <= s < 0.8),
        "high (80-100%)":    sum(1 for s in scores if s >= 0.8),
    }

c_dist = bucket(c_cons)
b_dist = bucket(b_cons)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (ARC-Challenge)")
print("=" * 65)
print(f"\n  CoT mean consistency     : {c_mean*100:.1f}%")
print(f"  Baseline mean consistency: {b_mean*100:.1f}%")
print(f"  Consistency lift         : {lift:.2f}x")
print(f"\n  CoT wins / Baseline wins / Tied: {cot_wins} / {baseline_wins} / {tied}")
print(f"\n  {'Bucket':<22} | {'CoT':>8} | {'Baseline':>8} | {'Diff':>6}")
for bkt in ["all_wrong (0%)", "low (1-39%)", "medium (40-79%)", "high (80-100%)"]:
    cv, bv = c_dist[bkt], b_dist[bkt]
    print(f"  {bkt:<22} | {cv:>8} | {bv:>8} | {cv-bv:>+6}")

angle2 = {
    "cot_mean_consistency": round(c_mean, 4),
    "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4),
    "cot_wins": cot_wins, "baseline_wins": baseline_wins, "tied": tied,
    "cot_distribution": c_dist, "baseline_distribution": b_dist,
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")

ANGLE 2 -- VOTE CONSISTENCY  (ARC-Challenge)

  CoT mean consistency     : 72.1%
  Baseline mean consistency: 71.5%
  Consistency lift         : 1.01x

  CoT wins / Baseline wins / Tied: 56 / 47 / 797

  Bucket                 |      CoT | Baseline |   Diff
  all_wrong (0%)         |      237 |      235 |     +2
  low (1-39%)            |        8 |       13 |     -5
  medium (40-79%)        |       16 |       19 |     -3
  high (80-100%)         |      639 |      633 |     +6

Saved -> /kaggle/working/arc_challenge_cot_eval/angle2_consistency.json


In [14]:
# CELL 14 -- ANGLE 3: CONFIDENCE CALIBRATION

def calibration_report(results, label):
    buckets = [
        ("Very High (>=0.80)",    lambda c: c >= 0.80, 0.90),
        ("High (0.60-0.80)",      lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium (0.40-0.60)",    lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low (<0.40)",           lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])
    print(f"\n  [{label}]")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset: continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"    {name:<22}: n={n}  acc={acc*100:.1f}%  expected={mid*100:.0f}%  gap={gap:.3f}  {flag}")
        calib_out.append({"bucket": name, "count": n, "accuracy": round(acc,4),
                          "expected": mid, "gap": round(gap,4)})
    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"    ECE={ece:.4f}  |  High-conf: {len(hc)}  |  Acc@high={hc_acc:.1f}%  |  False-conf: {false_conf}")
    return ece, calib_out, false_conf

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (ARC-Challenge)")
print("=" * 65)
c_ece, c_calib, c_false = calibration_report(cot_results,  "CoT")
b_ece, b_calib, b_false = calibration_report(base_results, "Baseline")

improve = (b_ece - c_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE: CoT={c_ece:.4f}  Baseline={b_ece:.4f}  Improvement={improve:.1f}%")
print(f"  False confidence: CoT={c_false}  Baseline={b_false}")

angle3 = {
    "cot_ece": round(c_ece, 4), "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(improve, 2),
    "cot_false_conf": c_false, "baseline_false_conf": b_false,
    "false_conf_reduction": b_false - c_false,
    "cot_calibration": c_calib, "baseline_calibration": b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")

ANGLE 3 -- CONFIDENCE CALIBRATION  (ARC-Challenge)

  [CoT]
    Very High (>=0.80)    : n=880  acc=72.6%  expected=90%  gap=0.174  Poor
    High (0.60-0.80)      : n=20  acc=50.0%  expected=70%  gap=0.200  Poor
    ECE=0.1744  |  High-conf: 880  |  Acc@high=72.6%  |  False-conf: 241

  [Baseline]
    Very High (>=0.80)    : n=870  acc=72.8%  expected=90%  gap=0.172  Poor
    High (0.60-0.80)      : n=28  acc=46.4%  expected=70%  gap=0.236  Poor
    Low (<0.40)           : n=2  acc=0.0%  expected=25%  gap=0.250  Poor
    ECE=0.1746  |  High-conf: 870  |  Acc@high=72.8%  |  False-conf: 237

  ECE: CoT=0.1744  Baseline=0.1746  Improvement=0.1%
  False confidence: CoT=241  Baseline=237

Saved -> /kaggle/working/arc_challenge_cot_eval/angle3_calibration.json


In [15]:
# CELL 15 -- ANGLE 4: ANSWER POSITION BIAS ANALYSIS
# ARC-Challenge-specific: does the model show preference for certain choice positions
# (A, B, C, D) regardless of which is actually correct?
#
# A well-calibrated model should perform equally when the correct answer is in
# any position. Systematic gaps reveal anchoring (e.g. always preferring A).
# CoT should reduce this by forcing the model to reason through all options.
#
# Position index: answer_pos = 0 -> A, 1 -> B, 2 -> C, 3 -> D

POS_LABEL = {0: "A", 1: "B", 2: "C", 3: "D"}


def position_bias_report(results, label):
    # Accuracy per correct-answer position
    by_pos = defaultdict(list)
    for r in results:
        pos = r.get("answer_pos", -1)
        if pos >= 0:
            by_pos[pos].append(r)

    pos_accs = {}
    print(f"\n  [{label}] -- Accuracy by correct-answer position")
    print(f"    {'Pos':<6} | {'N':>5} | {'Acc':>8}")
    for pos in sorted(by_pos):
        items = by_pos[pos]
        acc   = sum(r["correct"] for r in items) / len(items) * 100
        pos_accs[POS_LABEL.get(pos, str(pos))] = round(acc, 2)
        print(f"    {POS_LABEL.get(pos, str(pos)):<6} | {len(items):>5} | {acc:>7.1f}%")

    # Prediction distribution: how often model picks each letter
    pred_counts = Counter(r["final_answer"] for r in results)
    n = len(results)
    print(f"\n  [{label}] -- Model's prediction distribution")
    for ltr in ["A", "B", "C", "D"]:
        pct = pred_counts.get(ltr, 0) / n * 100
        bar = "#" * int(pct / 2)
        print(f"    {ltr}: {pct:>5.1f}%  {bar}")
    pct_fail = (n - sum(pred_counts.get(l, 0) for l in ["A","B","C","D"])) / n * 100
    print(f"    ?: {pct_fail:>5.1f}%  (extraction failures)")

    # Position bias score: std dev of per-position accuracy -- 0 = perfectly unbiased
    accs = list(pos_accs.values())
    bias_score = round(float(np.std(accs)), 4) if accs else 0.0
    print(f"\n  [{label}] Position bias score (std of per-pos acc): {bias_score:.4f}")
    print(f"           (0 = perfectly balanced, higher = more biased)")

    return {
        "accuracy_by_position": pos_accs,
        "prediction_distribution": {ltr: round(pred_counts.get(ltr, 0) / n * 100, 2) for ltr in ["A","B","C","D"]},
        "extraction_failure_pct": round(pct_fail, 2),
        "position_bias_score": bias_score,
    }


print("=" * 65)
print("ANGLE 4 -- ANSWER POSITION BIAS  (ARC-Challenge)")
print("=" * 65)
c_bias = position_bias_report(cot_results,  "CoT")
b_bias = position_bias_report(base_results, "Baseline")

bias_reduction = b_bias["position_bias_score"] - c_bias["position_bias_score"]
print(f"\n  Bias score: CoT={c_bias['position_bias_score']:.4f}  Baseline={b_bias['position_bias_score']:.4f}")
print(f"  Bias reduction from CoT: {bias_reduction:+.4f}")
if bias_reduction > 1.0:
    print("  RESULT: CoT meaningfully reduces position bias -- model evaluates all options.")
elif abs(bias_reduction) <= 1.0:
    print("  RESULT: Minimal bias change -- CoT does not alter position anchoring.")
else:
    print("  RESULT: CoT increases position bias -- unexpected, worth investigating.")

angle4 = {
    "dataset": "ARC-Challenge",
    "cot": c_bias,
    "baseline": b_bias,
    "bias_reduction": round(bias_reduction, 4),
}
with open(CONFIG["angle4_file"], "w") as f:
    json.dump(angle4, f, indent=2)
print(f"\nSaved -> {CONFIG['angle4_file']}")

ANGLE 4 -- ANSWER POSITION BIAS  (ARC-Challenge)

  [CoT] -- Accuracy by correct-answer position
    Pos    |     N |      Acc
    A      |   195 |    75.4%
    B      |   255 |    72.2%
    C      |   226 |    68.1%
    D      |   224 |    73.2%

  [CoT] -- Model's prediction distribution
    A:  24.9%  ############
    B:  27.6%  #############
    C:  21.4%  ##########
    D:  26.1%  #############
    ?:   0.0%  (extraction failures)

  [CoT] Position bias score (std of per-pos acc): 2.6275
           (0 = perfectly balanced, higher = more biased)

  [Baseline] -- Accuracy by correct-answer position
    Pos    |     N |      Acc
    A      |   195 |    80.5%
    B      |   255 |    69.4%
    C      |   226 |    69.5%
    D      |   224 |    69.2%

  [Baseline] -- Model's prediction distribution
    A:  29.4%  ##############
    B:  25.7%  ############
    C:  22.3%  ###########
    D:  22.6%  ###########
    ?:   0.0%  (extraction failures)

  [Baseline] Position bias score (std of p

In [16]:
# CELL 16 -- 3-WAY COMPARISON SUMMARY (Paper Table)
# Paste your Guided results from the original notebook below.

# ---- Paste guided results from original notebook ----
GUIDED_ACC     = 55.0   # guided accuracy %
GUIDED_ECE     = 0.100  # guided ECE
GUIDED_COMPUTE = 10.5   # B param-passes
# -----------------------------------------------------

c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
c_ece = json.load(open(CONFIG["angle3_file"]))["cot_ece"]
b_ece = json.load(open(CONFIG["angle3_file"]))["baseline_ece"]

c_bias_score = json.load(open(CONFIG["angle4_file"]))["cot"]["position_bias_score"]
b_bias_score = json.load(open(CONFIG["angle4_file"]))["baseline"]["position_bias_score"]

print("=" * 72)
print("  ARC-Challenge -- 3-WAY COMPARISON  (Paper Table)")
print("=" * 72)
print(f"  N={len(cot_results)}  Seed={CONFIG['random_seed']}  Solver=Qwen2.5-1.5B")
print()
print(f"  {'Condition':<30} | {'Compute':>7} | {'Accuracy':>9} | {'ECE':>7} | {'vs Baseline':>12}")
print(f"  {'-'*30}-+-{'-'*7}-+-{'-'*9}-+-{'-'*7}-+-{'-'*12}")
print(f"  {'Baseline (no guide, no CoT)':<30} | {'7.5B':>7} | {b_acc:>8.1f}% | {b_ece:>7.4f} | {'—':>12}")
print(f"  {'CoT (think step by step)':<30} | {'7.5B':>7} | {c_acc:>8.1f}% | {c_ece:>7.4f} | {c_acc-b_acc:>+11.1f}%")
print(f"  {'Guided (fine-tuned 3B guide)':<30} | {'10.5B':>7} | {GUIDED_ACC:>8.1f}% | {GUIDED_ECE:>7.4f} | {GUIDED_ACC-b_acc:>+11.1f}%")
print()
print(f"  POSITION BIAS (std of per-pos acc): CoT={c_bias_score:.4f}  Baseline={b_bias_score:.4f}")
print()
print(f"  KEY QUESTION: Does Guided beat CoT?")
print(f"  Guided vs CoT: {GUIDED_ACC - c_acc:+.1f} pts")
if GUIDED_ACC > c_acc + 2:
    print("  RESULT: Guided clearly outperforms CoT. Guide model justified.")
elif abs(GUIDED_ACC - c_acc) <= 2:
    print("  RESULT: Marginal difference (<2pts). Report honestly -- discuss tradeoff.")
else:
    print("  RESULT: CoT matches/beats Guided. Important finding to report honestly.")

  ARC-Challenge -- 3-WAY COMPARISON  (Paper Table)
  N=900  Seed=42  Solver=Qwen2.5-1.5B

  Condition                      | Compute |  Accuracy |     ECE |  vs Baseline
  -------------------------------+---------+-----------+---------+-------------
  Baseline (no guide, no CoT)    |    7.5B |     71.8% |  0.1746 |            —
  CoT (think step by step)       |    7.5B |     72.1% |  0.1744 |        +0.3%
  Guided (fine-tuned 3B guide)   |   10.5B |     55.0% |  0.1000 |       -16.8%

  POSITION BIAS (std of per-pos acc): CoT=2.6275  Baseline=4.8291

  KEY QUESTION: Does Guided beat CoT?
  Guided vs CoT: -17.1 pts
  RESULT: CoT matches/beats Guided. Important finding to report honestly.
